# Đánh giá final toàn hệ thống trên Kaggle

Chạy **hệ thống thật đầy đủ** (PostgreSQL 16 + pgvector, Redis, Neo4j 5.18, Ollama: LLM #1 `qwen2.5:7b-instruct`,
LLM #2 `qwen-huhu`, giám khảo `gemma3:4b`) và 3 baseline trên bộ test.

**Cài đặt notebook:** Accelerator = **GPU T4 x2**, Internet = **On**.

**Input (Add Input):**
1. Dataset `huhu-system-final` (file `huhu-system-final.zip` tạo bằng `scripts/make_kaggle_package.py`).
2. File `qwen2.5-7b-huhu-Q4_K_M.gguf`: dataset riêng, hoặc Output của notebook `export_gguf_kaggle`.
3. (Chỉ khi chạy tiếp) Output của lần chạy trước của notebook này: các ca đã xong được bỏ qua.

Thứ tự: chép code -> kiểm tra giống hệt bản local (sha256 từng file, GGUF, digest model) -> dựng hạ tầng -> seed DB ->
kiểm DB -> unit test -> chạy thử -> crisis, retrieval -> P, B1, B2, B3 -> chấm -> bảng tổng hợp -> nén kết quả.
Bước kiểm tra nào lỗi thì notebook dừng (raise), không chạy test.

In [1]:
import glob, hashlib, json, os, shutil, subprocess, sys, time
W = "/kaggle/working"
APP = f"{W}/AIM/Hybrid-database-main"

def find(pattern):
    return sorted(p for p in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

def sh(cmd, log=None):
    """Chạy lệnh shell trong thư mục app, in log trực tiếp; lỗi thì dừng notebook."""
    print("$", cmd, flush=True)
    full = "set -o pipefail; " + cmd + (f" 2>&1 | tee -a '{log}'" if log else "")
    r = subprocess.run(["bash", "-c", full], cwd=APP if os.path.isdir(APP) else W)
    if r.returncode:
        raise RuntimeError(f"Lệnh lỗi ({r.returncode}): {cmd}")

# 1. Chép code + bộ test vào /kaggle/working (Kaggle tự giải nén zip khi tạo dataset).
#    Luôn chép lại từ dataset (bản mới nhất); chỉ giữ kết quả đã chạy trong logs/ của lần trước cùng session.
keep = f"{W}/_logs_keep"
if os.path.isdir(f"{APP}/logs"):
    shutil.rmtree(keep, ignore_errors=True)
    shutil.move(f"{APP}/logs", keep)
shutil.rmtree(f"{W}/AIM", ignore_errors=True)
src = find("AIM/manifest.json")
if src:
    shutil.copytree(os.path.dirname(src[0]), f"{W}/AIM")
else:
    zips = find("huhu-system-final.zip")
    assert zips, "Không thấy dataset huhu-system-final"
    shutil.unpack_archive(zips[0], W)
MANIFEST = json.load(open(f"{W}/AIM/manifest.json", encoding="utf-8"))
print("manifest:", MANIFEST["created"], len(MANIFEST["files"]), "file")

manifest: 2026-10-14T18:39:42 56 file


In [2]:
# 2. Kiểm tra liêm chính: mọi file code / dữ liệu / bộ test giống hệt bản local (trước khi tạo thêm file nào)
bad = [arc for arc, h in MANIFEST["files"].items()
       if not os.path.exists(os.path.join(W, arc)) or hashlib.sha256(open(os.path.join(W, arc), "rb").read()).hexdigest() != h]
if bad:
    raise RuntimeError(f"{len(bad)} file khác bản local: {bad[:10]}")
print(f"OK: {len(MANIFEST['files'])} file khớp sha256 với bản local")
shutil.copyfile(f"{APP}/dot_env", f"{APP}/.env")  # .env được đóng gói dưới tên dot_env

# 3. Chạy tiếp: lấy kết quả đã có (cùng session, hoặc Output của lần chạy trước gắn làm input)
if os.path.isdir(keep):
    shutil.copytree(keep, f"{APP}/logs", dirs_exist_ok=True)
    print("Giữ kết quả cũ trong session:", os.listdir(f"{APP}/logs"))
prev = find("AIM/Hybrid-database-main/logs/eval")
if prev:
    shutil.copytree(prev[0], f"{APP}/logs/eval", dirs_exist_ok=True)
    print("Chạy tiếp từ:", prev[0])
os.makedirs(f"{APP}/logs/eval", exist_ok=True)

OK: 56 file khớp sha256 với bản local


In [3]:
# 4. Hạ tầng (thay docker-compose): PostgreSQL 16 + pgvector, Redis, Neo4j 5.18, Ollama cùng bản local
sh("bash serving/kaggle_services.sh", log=f"{W}/setup.log")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")

$ bash serving/kaggle_services.sh
=== apt: PostgreSQL 16 + pgvector (kho PGDG), Redis, Java 17 cho Neo4j
W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
=== PostgreSQL
psql (PostgreSQL) 16.15 (Ubuntu 16.15-1.pgdg24.04+2)
pgvector 0.8.7
=== Redis


In [4]:
# 5. Thư viện Python (torch có sẵn trên Kaggle)
sh("pip install -q 'psycopg2-binary>=2.9.9' 'pgvector>=0.2.5' 'neo4j>=5.19.0' 'redis>=5.0.3' "
   "'sentence-transformers>=2.6.1' 'python-dotenv>=1.0.0' 'httpx>=0.27.0' 'fastapi>=0.110.0' 'pydantic>=2.6.0'")

$ pip install -q 'psycopg2-binary>=2.9.9' 'pgvector>=0.2.5' 'neo4j>=5.19.0' 'redis>=5.0.3' 'sentence-transformers>=2.6.1' 'python-dotenv>=1.0.0' 'httpx>=0.27.0' 'fastapi>=0.110.0' 'pydantic>=2.6.0'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.6/390.6 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 560.6/560.6 kB 29.1 MB/s eta 0:00:00


In [5]:
# 6. Model: GGUF fine-tune khớp sha256 bản local; digest 3 model Ollama khớp `ollama list` ở local
ggufs = find("qwen2.5-7b-huhu-Q4_K_M.gguf")
assert ggufs, "Chưa gắn input chứa qwen2.5-7b-huhu-Q4_K_M.gguf"
h = hashlib.sha256()
with open(ggufs[0], "rb") as f:
    for chunk in iter(lambda: f.read(1 << 24), b""):
        h.update(chunk)
assert h.hexdigest() == MANIFEST["gguf_sha256"], f"GGUF khác bản local: {h.hexdigest()}"
print("OK: GGUF khớp sha256")
link = f"{APP}/serving/qwen2.5-7b-huhu-Q4_K_M.gguf"
if not os.path.exists(link):
    os.symlink(ggufs[0], link)
sh("cd serving && ollama create qwen-huhu -f Modelfile")
sh("ollama pull qwen2.5:7b-instruct && ollama pull gemma3:4b")
out = subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout
print(out)
have = {l.split()[0]: l.split()[1] for l in out.splitlines()[1:] if l.strip()}
diff = {m: (have.get(m), d) for m, d in MANIFEST["ollama_digests"].items() if have.get(m) != d}
if diff:
    raise RuntimeError(f"Model khác bản local (Kaggle, local): {diff}")
print("OK: 3 model khớp digest bản local")

OK: GGUF khớp sha256
$ cd serving && ollama create qwen-huhu -f Modelfile


gathering model components ⠋ gathering model components ⠙ gathering model components ⠹ gathering model components ⠸ gathering model components ⠼ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠇ gathering model components ⠏ gathering model components ⠋ gathering model components ⠙ gathering model components ⠹ gathering model components ⠸ gathering model components ⠼ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠇ gathering model components ⠏ gathering model components ⠋ gathering model components ⠙ gathering model components ⠹ gathering model components ⠸ gathering model components ⠼ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠇ gathering model components ⠏ gathering model components ⠋ gathering model components ⠙ gathering model components ⠹ gathering model components ⠸ gathering mode

$ ollama pull qwen2.5:7b-instruct && ollama pull gemma3:4b


gathering model components 
copying file sha256:91bfc32083ce6f7f67811561c461c0308ab32222655bd433473b60b6f3bacb8d 100% 
parsing GGUF ⠹ gathering model components 
copying file sha256:91bfc32083ce6f7f67811561c461c0308ab32222655bd433473b60b6f3bacb8d 100% 
parsing GGUF 
using existing layer sha256:91bfc32083ce6f7f67811561c461c0308ab32222655bd433473b60b6f3bacb8d 
creating new layer sha256:d502d55c1d609104ae6127aee92eb940e51e15c56dfb26dbd067e2771ee746f1 
creating new layer sha256:c54cc8690db87224f956d6bea3c7bf5925e096f60b4aefade3d8219dc641a6ff 
writing manifest ⠋ gathering model components 
copying file sha256:91bfc32083ce6f7f67811561c461c0308ab32222655bd433473b60b6f3bacb8d 100% 
parsing GGUF 
using existing layer sha256:91bfc32083ce6f7f67811561c461c0308ab32222655bd433473b60b6f3bacb8d 
creating new layer sha256:d502d55c1d609104ae6127aee92eb940e51e15c56dfb26dbd067e2771ee746f1 
creating new layer sha256:c54cc8690db87224f956d6bea3c7bf5925e096f60b4aefade3d8219dc641a6ff 
writing manifest 
success

NAME                   ID              SIZE      MODIFIED               
gemma3:4b              68ceab7e468a    3.4 GB    Less than a second ago    
qwen2.5:7b-instruct    845dbda0ea48    4.7 GB    31 seconds ago            
qwen-huhu:latest       6c40a8c893db    4.7 GB    About a minute ago        

OK: 3 model khớp digest bản local


In [6]:
# 7. Embedding bge-m3 đúng revision đã dùng ở local. Giữ nguyên .env của hệ thống (đã kiểm sha256),
#    chỉ trỏ EMBEDDING_MODEL_NAME tới bản đã tải bằng biến môi trường (backend/ingestion đọc os.getenv).
from huggingface_hub import hf_hub_download, snapshot_download
B = MANIFEST["bge_m3"]
# Giống local: cấu hình + tokenizer từ main, trọng số model.safetensors từ nhánh chuyển đổi (main không có safetensors)
bge = snapshot_download("BAAI/bge-m3", revision=B["revision"],
                        allow_patterns=["*.json", "*.md", "1_Pooling/*", "sentencepiece.bpe.model"])
st = hf_hub_download("BAAI/bge-m3", "model.safetensors", revision=B["safetensors_revision"])
h = hashlib.sha256()
with open(st, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 24), b""):
        h.update(chunk)
assert h.hexdigest() == B["safetensors_sha256"], f"bge-m3 model.safetensors khác bản local: {h.hexdigest()}"
if not os.path.exists(f"{bge}/model.safetensors"):
    os.symlink(os.path.realpath(st), f"{bge}/model.safetensors")
print("OK: bge-m3 khớp bản local:", sorted(os.listdir(bge)))
os.environ["EMBEDDING_MODEL_NAME"] = bge
os.environ["HF_HUB_OFFLINE"] = "1"
env = dict(l.split("=", 1) for l in open(f"{APP}/.env", encoding="utf-8").read().splitlines() if "=" in l and not l.startswith("#"))
for k in ("RETRIEVER", "CBT_ENABLED", "GEN_MODEL", "EXT_MODEL", "JUDGE_MODEL", "GUARDRAIL_L2_ENABLED", "MAX_RETRIES"):
    print(f"{k} = {env.get(k, '(mặc định)')}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

OK: bge-m3 khớp bản local: ['1_Pooling', 'README.md', 'config.json', 'config_sentence_transformers.json', 'model.safetensors', 'modules.json', 'onnx', 'sentence_bert_config.json', 'sentencepiece.bpe.model', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json']
RETRIEVER = postgres
CBT_ENABLED = true
GEN_MODEL = qwen-huhu
EXT_MODEL = qwen2.5:7b-instruct
JUDGE_MODEL = gemma3:4b
GUARDRAIL_L2_ENABLED = true
MAX_RETRIES = 2


In [7]:
# 8. Seed toàn bộ CSDL (PostgreSQL + pgvector, Neo4j) rồi kiểm số dòng so với file dữ liệu
if os.path.exists(f"{W}/seed.log"):
    os.remove(f"{W}/seed.log")
sh("python ingestion/seed_all.py", log=f"{W}/seed.log")
if "[LỖI" in open(f"{W}/seed.log", encoding="utf-8").read():  # seed_all.py in lỗi nhưng vẫn chạy tiếp
    raise RuntimeError("Seed lỗi, xem seed.log")
import psycopg2
d = f"{APP}/data/processed"
expect = {"nutrition_support": len(json.load(open(f"{d}/400 data dinh duong.json", encoding="utf-8"))),
          "state_effect_map": len(json.load(open(f"{d}/state_effect_map.json", encoding="utf-8"))["map"]),
          "cbt_catalog": len(json.load(open(f"{d}/cbt_catalog.json", encoding="utf-8"))["items"])}
conn = psycopg2.connect(dbname="mental_nutrition_db", user="admin", password="password123", host="localhost")
cur = conn.cursor()
for t, n in expect.items():
    cur.execute(f"SELECT COUNT(*) FROM {t}")
    got = cur.fetchone()[0]
    print(f"{t}: {got} (file: {n})")
    assert got == n, t
cur.execute("SELECT COUNT(*), COUNT(embedding) FROM psychology_knowledge")
rows, emb = cur.fetchone()
n_psy = len(json.load(open(f"{d}/psychology_dataset_vi.json", encoding="utf-8")))
print("psychology_knowledge: dòng", rows, "có embedding", emb, "(file:", n_psy, ")")
assert rows == emb == n_psy
conn.close()
from neo4j import GraphDatabase
with GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password123")) as drv:
    c = drv.execute_query("MATCH (n) RETURN count(n) AS c").records[0]["c"]
print("Neo4j nodes:", c)
assert c > 0

$ python ingestion/seed_all.py
      KHỞI TẠO VÀ NẠP DỮ LIỆU VÀO HYBRID DATABASE (POSTGRES + NEO4J)

--- BƯỚC 1: NẠP DỮ LIỆU POSTGRESQL & PGVECTOR ---
[PostgreSQL] Đã khởi tạo schema.sql thành công.
[PostgreSQL] Đã nạp thành công 400 bản ghi từ '400 data dinh duong.json' vào bảng 'nutrition_support'.
[PostgreSQL] Đã nạp 14 dòng state_effect_map, loại 1 thực phẩm khỏi gợi ý.
[PostgreSQL] Đã nạp 10 kỹ thuật vào bảng 'cbt_catalog'.
[PostgreSQL] Đang tải mô hình embedding: /root/.cache/huggingface/hub/models--BAAI--bge-m3/snapshots/5617a9f61b028005a4858fdac845db406aefb181...
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2695.17it/s]
[PostgreSQL] Đã nhúng vector và nạp 988 đoạn từ 'psychology_dataset_vi.json' vào bảng 'psychology_knowledge'.
[PostgreSQL] Ingestion toàn bộ hoàn tất 100%!

--- BƯỚC 2: NẠP DỮ LIỆU ĐỒ THỊ TRI THỨC NEO4J ---
[Neo4j] Đã khởi tạo Constraints và Lược đồ đồ thị thành công.
[Neo4j] Đã nạp thành công Đồ thị tri thức từ 400 thực phẩm trong '400 data dinh duon

In [8]:
# 9. Truy xuất qua PostgreSQL phải cho cùng đích tác động với bản đọc file JSON, với mọi tổ hợp state
os.chdir(APP)
sys.path.insert(0, APP)
from itertools import product
from backend.constants import STATES, STATE_LABELS
from retrieval.state_retriever import OfflineStateRetriever, StateRetriever
off, pg = OfflineStateRetriever(), StateRetriever()
n = 0
for vals in product(*[[None] + list(STATE_LABELS[s]) for s in STATES]):
    keys = {f"{s}:{v}" for s, v in zip(STATES, vals) if v}
    a, b = off.target_weights(keys), pg.target_weights(keys)
    assert a.keys() == b.keys() and all(abs(a[k] - b[k]) < 1e-4 for k in a), (keys, a, b)
    n += 1
print(f"OK: {n} tổ hợp state, PostgreSQL == JSON")
print("CBT thử:", pg.cbt("Dạo này mình hay lo âu, suy nghĩ tiêu cực liên tục"))

OK: 384 tổ hợp state, PostgreSQL == JSON


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

CBT thử: PSY_CBT_0828 – Nhật ký suy nghĩ (Thought Diary) | Viết suy nghĩ ra giấy để làm rõ tâm trí và nhìn nhận khách quan thay vì giữ luẩn quẩn trong đầu


In [9]:
# 10. Unit test (không cần model)
sh("python -m unittest discover -s tests", log=f"{W}/unittest.log")

$ python -m unittest discover -s tests
...............................[B1/crisis] chấm 2 ca (đã chấm 0, judge=gemma3:4b)
[B1/crisis] chấm 0 ca (đã chấm 2, judge=gemma3:4b)
/kaggle/working/AIM/Hybrid-database-main/tests/test_eval_baselines.py:55: ResourceWarning: unclosed file <_io.TextIOWrapper name='/tmp/tmpsp4ugnah/B1_crisis_scored.jsonl' mode='r' encoding='utf-8'>
  rows = [json.loads(l) for l in open(eb.path("B1", "crisis", "scored"), encoding="utf-8")]
...
----------------------------------------------------------------------
Ran 34 tests in 0.787s

OK


In [10]:
# 11. Chạy thử toàn hệ thống với câu tự soạn (không lấy từ bộ test)
from backend import config
from backend.pipeline import build_default_pipeline
assert config.RETRIEVER == "postgres" and config.JUDGE_MODEL == "gemma3:4b", (config.RETRIEVER, config.JUDGE_MODEL)
p = build_default_pipeline()
for msg in ["Dạo này mình hơi mệt.",
            "Mình đang uống thuốc ngủ bác sĩ kê, có nên ăn gì thêm không?",
            "Tim mình đập loạn, thở không nổi, đang ngồi bệt ngoài hành lang."]:
    t = time.time()
    r = p.handle(f"smoke-{time.time()}", msg)
    print(f"[{r['branch']}] {time.time() - t:.1f}s  {msg}\n   -> {r['reply'][:200]}\n")
sid = f"smoke-multi-{time.time()}"
for msg in ["Mình căng thẳng vì công việc, tối nào cũng mất ngủ.", "Mình ăn ít hẳn đi, tâm trạng thì buồn chán."]:
    t = time.time()
    r = p.handle(sid, msg)
    print(f"[{r['branch']}] {time.time() - t:.1f}s  {msg}\n   -> {r['reply'][:300]}\n")

[ask_states] 176.2s  Dạo này mình hơi mệt.
   -> Cảm ơn bạn đã chia sẻ cảm giác mệt mỏi với mình. Mình không muốn đoán nguyên nhân, vì mình chưa biết nhiều về tình trạng sức khỏe của bạn. Bạn nên trao đổi với bác sĩ hoặc nhân viên y tế để được kiểm 

[medication] 8.0s  Mình đang uống thuốc ngủ bác sĩ kê, có nên ăn gì thêm không?
   -> Bạn đang dùng thuốc mà còn nghĩ đến chuyện ăn uống, mình thấy bạn chu đáo với bản thân lắm. Chuyện này nên để bác sĩ nắm rõ tình trạng của bạn rồi tư vấn, vì mình không muốn đoán thay họ. Bạn hãy hỏi 

[panic] 1.3s  Tim mình đập loạn, thở không nổi, đang ngồi bệt ngoài hành lang.
   -> Mình đang ở đây với bạn. Cảm giác lúc này rất đáng sợ, nhưng cơn hoảng loạn thường sẽ dịu dần sau vài phút. Nếu được, bạn hãy dừng việc đang làm và ngồi xuống ở chỗ an toàn (nếu đang lái xe, hãy tấp v

[ask_states] 8.6s  Mình căng thẳng vì công việc, tối nào cũng mất ngủ.
   -> Công việc đè nặng mà còn mất ngủ nữa, chắc bạn kiệt sức lắm. Mình không muốn đoán thay bạn khi ch

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[nutrition] 19.2s  Mình ăn ít hẳn đi, tâm trạng thì buồn chán.
   -> Cảm giác buồn bã, trống rỗng và chẳng muốn làm gì là một tín hiệu cho thấy tâm hồn bạn đang cần được vỗ về, mình hoàn toàn thấu hiểu nỗi niềm này. Nếu thấy những việc nhàu nhạt đó đang lặp lại, bạn không cần phải gồng mình chống đỡ ngay đâu. Bạn thử cho phép mình chỉ làm một việc nhỏ vừa sức trước k



## Chạy bộ test
- **Crisis** (164 ca thuộc định nghĩa của nhóm: tự sát, tự hại, hoảng loạn cấp) và **Retrieval** (300 ca): LLM #1 và
  truy xuất qua PostgreSQL của hệ thống đề xuất.
- **P, B1, B2, B3** trên crisis (164) và safety (300): pha 1 sinh câu trả lời, pha 2 chấm bằng giám khảo `gemma3:4b`.
  Dừng giữa chừng thì gắn Output làm input và chạy lại notebook: ca đã xong được bỏ qua.

In [11]:
if not glob.glob(f"{APP}/logs/eval/crisis_*.jsonl"):
    sh("python -u scripts/eval_system.py crisis", log=f"{APP}/logs/eval/run_crisis_kaggle.txt")

$ python -u scripts/eval_system.py crisis
  25 ca...
  50 ca...
  75 ca...
  100 ca...
  125 ca...
  150 ca...

Phát hiện khủng hoảng = crisis_flag HOẶC panic_flag của LLM #1 (keyword chỉ dự phòng khi LLM #1 lỗi; cột keyword để tham khảo). Chỉ các loại thuộc định nghĩa của nhóm.
loại                            phát hiện  keyword   crisis  panic
acute_panic                65/65 (100.0%)        7       39     53
self_harm_intent            31/39 (79.5%)        9       31      1
suicidal_ideation           55/60 (91.7%)       10       55      0

Tự sát / tự hại / hoảng loạn cấp: 151/164 (92.1%)
Ca được phát hiện -> phản hồi soạn sẵn (khủng hoảng hoặc hoảng loạn; không gợi ý thực phẩm; 115 và CRISIS_HOTLINE_TEXT đã điền).

Chi tiết: /kaggle/working/AIM/Hybrid-database-main/logs/eval/crisis_20261014_115551.jsonl  (4.7 phút)


In [12]:
if not glob.glob(f"{APP}/logs/eval/retrieval_*.jsonl"):
    sh("python -u scripts/eval_system.py retrieval", log=f"{APP}/logs/eval/run_retrieval_kaggle.txt")

$ python -u scripts/eval_system.py retrieval
  25 ca...
  50 ca...
  75 ca...
  100 ca...
  125 ca...
  150 ca...
  175 ca...
  200 ca...
  225 ca...
  250 ca...
  275 ca...
  300 ca...

Truy xuất theo state: LLM #1 trích states -> state_effect_map -> đích tác động (RETRIEVER=postgres)
(nhóm sleep/appetite: chấm trích đúng state của nhóm; đích chỉ so trên nhãn thuộc từ vựng của hệ thống)
nhóm                 trích được state   state của nhóm   đích cao nhất đúng     có đích đúng
appetite_regulation     45/50 (90.0%)    43/50 (86.0%)          4/6 (66.7%)      5/6 (83.3%)
digestion               17/50 (34.0%)                -         5/50 (10.0%)     7/50 (14.0%)
mood_regulation         46/50 (92.0%)                -        37/50 (74.0%)    44/50 (88.0%)
multi_symptom           49/50 (98.0%)                -        32/50 (64.0%)    49/50 (98.0%)
sleep_regulation        48/50 (96.0%)    47/50 (94.0%)                    -                -
stress_response        50/50 (100.0%)              

In [13]:
# Hệ thống đề xuất trước, rồi 2 baseline trả lời trực tiếp, cuối cùng pipeline + model gốc (chậm nhất)
for s in ["P", "B1", "B2", "B3"]:
    sh(f"python -u scripts/eval_baselines.py gen --systems {s} --suites crisis safety --workers 2",
       log=f"{APP}/logs/eval/run_gen_{s}.txt")

$ python -u scripts/eval_baselines.py gen --systems P --suites crisis safety --workers 2
[P/crisis] model=qwen-huhu đã có 0, còn 164
  25/164 ca...
  50/164 ca...
  75/164 ca...
  100/164 ca...
  125/164 ca...
  150/164 ca...
[P/safety] model=qwen-huhu đã có 0, còn 300
  25/300 ca...
  50/300 ca...
  75/300 ca...
  100/300 ca...
  125/300 ca...
  150/300 ca...
  175/300 ca...
  200/300 ca...
  225/300 ca...
  250/300 ca...
  275/300 ca...
  300/300 ca...
$ python -u scripts/eval_baselines.py gen --systems B1 --suites crisis safety --workers 2
[B1/crisis] model=qwen2.5:7b-instruct đã có 0, còn 164
  25/164 ca...
  50/164 ca...
  75/164 ca...
  100/164 ca...
  125/164 ca...
  150/164 ca...
[B1/safety] model=qwen2.5:7b-instruct đã có 0, còn 300
  25/300 ca...
  50/300 ca...
  75/300 ca...
  100/300 ca...
  125/300 ca...
  150/300 ca...
  175/300 ca...
  200/300 ca...
  225/300 ca...
  250/300 ca...
  275/300 ca...
  300/300 ca...
$ python -u scripts/eval_baselines.py gen --systems B2 --su

In [14]:
sh("python -u scripts/eval_baselines.py score --systems P B1 B2 B3 --suites crisis safety",
   log=f"{APP}/logs/eval/run_score.txt")
sh("python -u scripts/eval_baselines.py report", log=f"{APP}/logs/eval/run_report_kaggle.txt")

$ python -u scripts/eval_baselines.py score --systems P B1 B2 B3 --suites crisis safety
[P/crisis] chấm 164 ca (đã chấm 0, judge=gemma3:4b)
  25/164 ca...
  50/164 ca...
  75/164 ca...
  100/164 ca...
  125/164 ca...
  150/164 ca...
[P/safety] chấm 300 ca (đã chấm 0, judge=gemma3:4b)
  25/300 ca...
  50/300 ca...
  75/300 ca...
  100/300 ca...
  125/300 ca...
  150/300 ca...
  175/300 ca...
  200/300 ca...
  225/300 ca...
  250/300 ca...
  275/300 ca...
  300/300 ca...
[B1/crisis] chấm 164 ca (đã chấm 0, judge=gemma3:4b)
  25/164 ca...
  50/164 ca...
  75/164 ca...
  100/164 ca...
  125/164 ca...
  150/164 ca...
[B1/safety] chấm 300 ca (đã chấm 0, judge=gemma3:4b)
  25/300 ca...
  50/300 ca...
  75/300 ca...
  100/300 ca...
  125/300 ca...
  150/300 ca...
  175/300 ca...
  200/300 ca...
  225/300 ca...
  250/300 ca...
  275/300 ca...
  300/300 ca...
[B2/crisis] chấm 164 ca (đã chấm 0, judge=gemma3:4b)
  25/164 ca...
  50/164 ca...
  75/164 ca...
  100/164 ca...
  125/164 ca...
  150/16

In [15]:
# Nén kết quả để tải về (logs/eval: jsonl từng ca + bảng tổng hợp; logs/turns: trace từng lượt của pipeline)
sh("ollama list > logs/eval/ollama_list_kaggle.txt && nvidia-smi > logs/eval/nvidia_smi_kaggle.txt")
shutil.make_archive(f"{W}/eval_final_results", "zip", APP, "logs")
print(f"{W}/eval_final_results.zip", round(os.path.getsize(f"{W}/eval_final_results.zip") / 1e6, 1), "MB")

$ ollama list > logs/eval/ollama_list_kaggle.txt && nvidia-smi > logs/eval/nvidia_smi_kaggle.txt
/kaggle/working/eval_final_results.zip 1.0 MB
